# Assignment 5 - Perceptron
## Part 1: Heuristic Approach
## Part 2: Gradient Descent Approach

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Load and plot the data

In [ ]:
# load data
data = pd.read_csv('data.csv', header=None)
data.columns = ['x1', 'x2', 'label']

X = data[['x1', 'x2']].values
y = data['label'].values

# plot it
plt.figure(figsize=(6,5))
plt.scatter(X[y==1, 0], X[y==1, 1], color='blue', label='class 1')
plt.scatter(X[y==0, 0], X[y==0, 1], color='red', label='class 0')
plt.title('Data Plot')
plt.xlabel('x1')
plt.ylabel('x2')
plt.legend()
plt.show()

## Helper function: draw the separation line

In [ ]:
# given weights and bias, draw the line w1*x1 + w2*x2 + b = 0
# which means x2 = -(w1*x1 + b) / w2
def draw_line(w1, w2, b, color='green', linestyle='--', alpha=0.5):
    x1_vals = np.linspace(0, 1, 100)
    if w2 != 0:
        x2_vals = -(w1 * x1_vals + b) / w2
        plt.plot(x1_vals, x2_vals, color=color, linestyle=linestyle, alpha=alpha)

## Part 1 - Heuristic Perceptron

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def perceptron_heuristic(X, y, learning_rate=0.1, epochs=65):
    # random init
    np.random.seed(42)
    w = np.random.randn(2)
    b = np.random.randn()

    lines = []  # store all (w, b) after each epoch
    initial_w = w.copy()
    initial_b = b

    for epoch in range(epochs):
        for i in range(len(X)):
            z = np.dot(w, X[i]) + b
            y_hat = sigmoid(z)
            error = y[i] - y_hat
            b = b + learning_rate * error
            w = w + learning_rate * error * X[i]
        lines.append((w.copy(), b))

    return initial_w, initial_b, lines

def plot_part1(learning_rate=0.1, epochs=65):
    init_w, init_b, lines = perceptron_heuristic(X, y, learning_rate, epochs)

    plt.figure(figsize=(6,5))
    plt.scatter(X[y==1, 0], X[y==1, 1], color='blue', label='class 1')
    plt.scatter(X[y==0, 0], X[y==0, 1], color='red', label='class 0')

    # draw initial line in red
    draw_line(init_w[0], init_w[1], init_b, color='red', linestyle='-', alpha=1.0)

    # draw middle lines in dashed green
    for w, b in lines[:-1]:
        draw_line(w[0], w[1], b, color='green', linestyle='--', alpha=0.4)

    # draw last line in black
    final_w, final_b = lines[-1]
    draw_line(final_w[0], final_w[1], final_b, color='black', linestyle='-', alpha=1.0)

    plt.title(f'Part 1 - Heuristic (lr={learning_rate}, epochs={epochs})')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.legend()
    plt.show()

In [ ]:
# try different learning rates
plot_part1(learning_rate=0.01, epochs=65)
plot_part1(learning_rate=0.1, epochs=65)
plot_part1(learning_rate=1, epochs=65)

## Part 2 - Gradient Descent Perceptron

In [ ]:
def log_loss(X, y, w, b):
    total = 0
    for i in range(len(X)):
        z = np.dot(w, X[i]) + b
        y_hat = sigmoid(z)
        # add small number so we dont get log(0)
        y_hat = np.clip(y_hat, 1e-10, 1 - 1e-10)
        total += -(y[i] * np.log(y_hat) + (1 - y[i]) * np.log(1 - y_hat))
    return total / len(X)

def perceptron_gradient(X, y, learning_rate=0.1, epochs=100):
    np.random.seed(42)
    w = np.random.randn(2)
    b = np.random.randn()

    initial_w = w.copy()
    initial_b = b

    lines = []
    errors = []
    error_epochs = []

    for epoch in range(epochs):
        for i in range(len(X)):
            z = np.dot(w, X[i]) + b
            y_hat = sigmoid(z)
            # use 0.5 as threshold to classify
            pred = 1 if y_hat >= 0.5 else 0
            if pred == 0 and y[i] == 1:
                b = b + learning_rate
                w = w + learning_rate * X[i]
            elif pred == 1 and y[i] == 0:
                b = b - learning_rate
                w = w - learning_rate * X[i]

        lines.append((w.copy(), b))

        # record error every 10 epochs
        if (epoch + 1) % 10 == 0 or epoch == 0:
            err = log_loss(X, y, w, b)
            errors.append(err)
            error_epochs.append(epoch + 1)

    return initial_w, initial_b, lines, errors, error_epochs


def plot_part2(learning_rate=0.1, epochs=100):
    init_w, init_b, lines, errors, error_epochs = perceptron_gradient(X, y, learning_rate, epochs)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # left plot - solution boundary
    plt.sca(axes[0])
    plt.scatter(X[y==1, 0], X[y==1, 1], color='blue', label='class 1')
    plt.scatter(X[y==0, 0], X[y==0, 1], color='red', label='class 0')

    draw_line(init_w[0], init_w[1], init_b, color='red', linestyle='-', alpha=1.0)

    for w, b in lines[:-1]:
        draw_line(w[0], w[1], b, color='green', linestyle='--', alpha=0.3)

    final_w, final_b = lines[-1]
    draw_line(final_w[0], final_w[1], final_b, color='black', linestyle='-', alpha=1.0)

    plt.title(f'Part 2 - Gradient Descent (lr={learning_rate})')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.legend()

    # right plot - error graph
    plt.sca(axes[1])
    plt.plot(error_epochs, errors, color='blue')
    plt.title('Error Plot')
    plt.xlabel('Number of epochs')
    plt.ylabel('Log Loss')

    plt.tight_layout()
    plt.show()

In [ ]:
# try different settings
plot_part2(learning_rate=0.01, epochs=100)
plot_part2(learning_rate=0.1, epochs=100)
plot_part2(learning_rate=1, epochs=100)